# Python challenges 3: stacks and monotonic stacks

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/python/practice/python-challenges-03-stacks-and-monotonic-stacks.ipynb)

A stack keeps the most recent unresolved item on top. Several classic problems become simple once you see that. Statements are paraphrased, and each solution is collapsed under its exercise.

<details>
<summary><b>How to use this notebook</b></summary>

Work through the exercises in order. Each one has three parts:

1. **Question:** what to build, with the rules it must follow.
2. **Exercise cell:** replace the `pass` with your code. Keep the function name and parameters.
3. **Check cell:** run it after your code. Each line says `pass` when your function gives the expected result, and `todo` when it does not yet. Checks never stop the notebook, so you can run everything at any time.

The last cell prints your score. Use `Runtime -> Run all` to rerun everything after a change.

</details>

In [ ]:
_tally = {"passed": 0, "total": 0}

def check(label, make_actual, expected):
    """Run one check. It never raises, so the notebook always runs from top to bottom."""
    _tally["total"] += 1
    try:
        actual = make_actual()
    except Exception as error:
        print(f"error  {label}: {type(error).__name__}: {error}")
        return
    if actual == expected:
        _tally["passed"] += 1
        print(f"pass   {label}")
    else:
        print(f"todo   {label}: got {actual!r}")

def raises(make_actual, error_type):
    """True when the call raises error_type, False when it raises anything else or nothing."""
    try:
        make_actual()
    except error_type:
        return True
    except Exception:
        return False
    return False

def score():
    print(f"\nScore: {_tally['passed']} of {_tally['total']} checks pass")

## Exercise 1: Valid Parentheses

**Difficulty:** Easy · about 10 min

Given a string made of `()[]{}`, return `True` when every bracket is closed by the right kind of bracket in the right order.

In a pipeline: the same stack check confirms that brackets and braces are balanced in a hand-written parser for config files or JSON-like log lines.

**Example 1**

```text
Input:  brackets = "()[]{}"
Output: True
```

**Example 2**

```text
Input:  brackets = "(]"
Output: False
```

<details>
<summary>💡 Tip</summary>

Push each opening bracket. When a closing bracket arrives, the top of the stack must be its matching opener, or the string is invalid. The stack must be empty at the end, which catches unclosed openers.

</details>

In [ ]:
def is_valid(brackets):
    # your code here
    pass

In [ ]:
check("is_valid('()[]{}')", lambda: is_valid("()[]{}"), True)
check("is_valid('(]')", lambda: is_valid("(]"), False)
check("is_valid('([)]')", lambda: is_valid("([)]"), False)
check("is_valid('{[]}')", lambda: is_valid("{[]}"), True)
check("empty string is valid", lambda: is_valid(""), True)
check("unclosed opening: '('", lambda: is_valid("("), False)
check("closing with nothing open: ')'", lambda: is_valid(")"), False)
check("closing before opening: '){'", lambda: is_valid("){"), False)
check("deep nesting of one kind", lambda: is_valid("((()))"), True)
check("long valid input (30,000 brackets)", lambda: is_valid("([{" * 5_000 + "}])" * 5_000), True)

<details>
<summary><b>Show solution</b></summary>

```python
def is_valid(brackets):
    # Approach: a stack holds the opening brackets still waiting for their partner.
    # A closing bracket must match the most recent opening one, which is the stack top.
    # Complexity: O(n) time, O(n) space.
    pairs = {")": "(", "]": "[", "}": "{"}     # closing bracket -> the opening it needs
    stack = []
    for char in brackets:
        if char in pairs:                        # closing bracket
            if not stack or stack.pop() != pairs[char]:
                return False                     # nothing open, or the wrong kind is open
        else:
            stack.append(char)                   # opening bracket: wait for its partner
    return not stack                             # anything left open is unmatched
```

</details>

## Exercise 2: Min Stack

**Difficulty:** Medium · about 15 min

Write a class `MinStack` with four operations that all run in O(1):

- `push(value)` adds a value.
- `pop()` removes the top value and returns it.
- `top()` returns the top value without removing it.
- `get_min()` returns the smallest value currently in the stack.

**Example 1**

```text
Input:  push(-2), push(0), push(-3), get_min(), pop(), top(), get_min()
Output: [None, None, None, -3, -3, 0, -2]
```

**Example 2**

```text
Input:  push(1), push(1), pop(), get_min()
Output: [None, None, 1, 1]
```

<details>
<summary>💡 Tip</summary>

Keep a second stack of minimums. Each push also pushes the smaller of the new value and the current minimum, so the top of the second stack is always the answer for get_min. Pop from both stacks together.

</details>

In [ ]:
class MinStack:
    # your code here
    pass

In [ ]:
def run_min_stack(operations):
    """Apply (method, *args) steps to a fresh MinStack and collect each return value."""
    stack = MinStack()
    return [getattr(stack, name)(*args) for name, *args in operations]

check(
    "MinStack get_min, pop, top, get_min",
    lambda: (lambda s: (s.push(-2), s.push(0), s.push(-3), s.get_min(), s.pop(), s.top(), s.get_min())[3:])(MinStack()),
    (-3, -3, 0, -2),
)
check(
    "duplicate minimums survive a pop",
    lambda: run_min_stack([("push", 1), ("push", 1), ("pop",), ("get_min",)]),
    [None, None, 1, 1],
)
check(
    "minimum restored after popping it",
    lambda: run_min_stack([("push", 5), ("push", 3), ("pop",), ("get_min",)]),
    [None, None, 3, 5],
)
check(
    "increasing pushes keep the first minimum",
    lambda: run_min_stack([("push", 1), ("push", 2), ("push", 3), ("get_min",), ("pop",), ("pop",), ("get_min",)]),
    [None, None, None, 1, 3, 2, 1],
)
check(
    "zero and negatives",
    lambda: run_min_stack([("push", 0), ("push", -1), ("push", 0), ("get_min",), ("pop",), ("get_min",)]),
    [None, None, None, -1, 0, -1],
)
check("single value", lambda: run_min_stack([("push", 7), ("top",), ("get_min",)]), [None, 7, 7])
check(
    "two stacks do not share state",
    lambda: (lambda first, second: (first.push(1), second.push(5), second.get_min())[2])(MinStack(), MinStack()),
    5,
)
check(
    "large input (100,000 pushes, 50,000 pops)",
    lambda: run_min_stack([("push", value) for value in range(100_000, 0, -1)] + [("pop",)] * 50_000 + [("get_min",)])[-1],
    50_001,
)

<details>
<summary><b>Show solution</b></summary>

```python
class MinStack:
    # Approach: each stack entry is (value, minimum so far at that depth).
    # The minimum of the whole stack is stored at the top, so get_min is O(1), and popping
    # restores the previous minimum automatically with no searching.
    # Complexity: every operation O(1) time, O(n) space.
    def __init__(self):
        self._items = []

    def push(self, value):
        lowest = min(value, self.get_min()) if self._items else value
        self._items.append((value, lowest))

    def pop(self):
        return self._items.pop()[0]

    def top(self):
        return self._items[-1][0]

    def get_min(self):
        return self._items[-1][1]
```

</details>

## Exercise 3: Daily Temperatures

**Difficulty:** Medium · about 20 min

For each day, return how many days you must wait for a warmer temperature. Return `0` when no warmer day follows.

In a pipeline: this is the next-greater-element pattern, for example how many ticks until a price beats today's, found in one pass instead of a self-join.

**Example 1**

```text
Input:  temps = [73, 74, 75, 71, 69, 72, 76, 73]
Output: [1, 1, 4, 2, 1, 1, 0, 0]
```

**Example 2**

```text
Input:  temps = [30, 40, 50, 60]
Output: [1, 1, 1, 0]
```

<details>
<summary>💡 Tip</summary>

Keep a stack of day indexes still waiting for a warmer day. When a warmer temperature arrives, pop every earlier day it is warmer than and record the distance between the two indexes. Days left on the stack get 0.

</details>

In [ ]:
def daily_temperatures(temps):
    # your code here
    pass

In [ ]:
check(
    "daily_temperatures([73, 74, 75, 71, 69, 72, 76, 73])",
    lambda: daily_temperatures([73, 74, 75, 71, 69, 72, 76, 73]),
    [1, 1, 4, 2, 1, 1, 0, 0],
)
check("rising temperatures", lambda: daily_temperatures([30, 40, 50, 60]), [1, 1, 1, 0])
check("falling temperatures: never warmer", lambda: daily_temperatures([60, 50, 40]), [0, 0, 0])
check("equal is not warmer", lambda: daily_temperatures([50, 50, 50]), [0, 0, 0])
check("tie then warmer: [70, 70, 71]", lambda: daily_temperatures([70, 70, 71]), [2, 1, 0])
check("negatives", lambda: daily_temperatures([-5, -10, 0]), [2, 1, 0])
check("single day", lambda: daily_temperatures([70]), [0])
check("empty input", lambda: daily_temperatures([]), [])
check("longer falling input (3,000 days)", lambda: daily_temperatures(list(range(3_000, 0, -1))), [0] * 3_000)

<details>
<summary><b>Show solution</b></summary>

```python
def daily_temperatures(temps):
    # Approach: a monotonic stack of day indexes whose temperature is still waiting for a
    # warmer day. The stack stays in decreasing temperature order. When a warmer day
    # arrives, it resolves every waiting day that is colder than it.
    # Complexity: O(n) time (each index is pushed and popped once), O(n) space.
    waits = [0] * len(temps)                     # 0 means "no warmer day" by default
    stack = []                                   # indexes of days still waiting
    for day, temp in enumerate(temps):
        while stack and temps[stack[-1]] < temp:
            earlier = stack.pop()
            waits[earlier] = day - earlier       # the gap in days is the answer
        stack.append(day)
    return waits
```

</details>

## Exercise 4: Evaluate Reverse Polish Notation

**Difficulty:** Medium · about 15 min

`tokens` is an expression in postfix form: numbers and the operators `+ - * /`. Evaluate it and return the integer result. Division truncates toward zero.

**Example 1**

```text
Input:  tokens = ["2", "1", "+", "3", "*"]
Output: 9
```

**Example 2**

```text
Input:  tokens = ["4", "13", "5", "/", "+"]
Output: 6
```

<details>
<summary>💡 Tip</summary>

Push numbers. When an operator arrives, pop two numbers: the first popped is the right operand. Division must truncate toward zero, which `int(a / b)` does for these inputs.

</details>

In [ ]:
def eval_rpn(tokens):
    # your code here
    pass

In [ ]:
check("eval_rpn(['2', '1', '+', '3', '*'])", lambda: eval_rpn(["2", "1", "+", "3", "*"]), 9)
check("eval_rpn(['4', '13', '5', '/', '+'])", lambda: eval_rpn(["4", "13", "5", "/", "+"]), 6)
check("eval_rpn(['-7', '2', '/'])", lambda: eval_rpn(["-7", "2", "/"]), -3)
check("single number", lambda: eval_rpn(["42"]), 42)
check("negative number token", lambda: eval_rpn(["3", "-4", "+"]), -1)
check("operand order for -", lambda: eval_rpn(["5", "3", "-"]), 2)
check("operand order for /", lambda: eval_rpn(["3", "6", "/"]), 0)
check("truncate toward zero, negative result", lambda: eval_rpn(["-6", "4", "/"]), -1)
check("multiply by zero", lambda: eval_rpn(["0", "5", "*"]), 0)
check(
    "longer expression",
    lambda: eval_rpn(["10", "6", "9", "3", "+", "-11", "*", "/", "*", "17", "+", "5", "+"]),
    22,
)

<details>
<summary><b>Show solution</b></summary>

```python
def eval_rpn(tokens):
    # Approach: in postfix notation an operator applies to the two most recent values.
    # Push numbers; on an operator, pop the right operand first (it was pushed last).
    # Complexity: O(n) time, O(n) space.
    stack = []
    for token in tokens:
        if token in "+-*/" and len(token) == 1:  # the length check keeps "-7" as a number
            right, left = stack.pop(), stack.pop()   # order matters for - and /
            if token == "+":
                stack.append(left + right)
            elif token == "-":
                stack.append(left - right)
            elif token == "*":
                stack.append(left * right)
            else:
                stack.append(int(left / right))  # truncate toward zero, as the problem asks
        else:
            stack.append(int(token))
    return stack[0]
```

</details>

## Exercise 5: Largest Rectangle in Histogram

**Difficulty:** Hard · about 35 min

`heights` gives the bar heights of a histogram, where every bar is one unit wide. Return the area of the largest rectangle that fits under the bars.

In a pipeline: the same stack finds the best sustained level in a metric series, the stretch that maximises its length times the lowest value inside it.

**Example 1**

```text
Input:  heights = [2, 1, 5, 6, 2, 3]
Output: 10
```

**Example 2**

```text
Input:  heights = [2, 4]
Output: 4
```

<details>
<summary>💡 Tip</summary>

Keep a stack of bar indexes with increasing heights. When a shorter bar arrives, each taller bar on the stack has found its right edge, so pop it and compute its area. Its left edge is the index left on the stack. Append a 0 at the end to flush the stack.

</details>

In [ ]:
def largest_rectangle(heights):
    # your code here
    pass

In [ ]:
check("largest_rectangle([2, 1, 5, 6, 2, 3])", lambda: largest_rectangle([2, 1, 5, 6, 2, 3]), 10)
check("largest_rectangle([2, 4])", lambda: largest_rectangle([2, 4]), 4)
check("empty histogram", lambda: largest_rectangle([]), 0)
check("single bar", lambda: largest_rectangle([5]), 5)
check("all equal bars", lambda: largest_rectangle([3, 3, 3, 3]), 12)
check("increasing bars", lambda: largest_rectangle([1, 2, 3, 4, 5]), 9)
check("decreasing bars", lambda: largest_rectangle([5, 4, 3, 2, 1]), 9)
check("zero-height bar splits the histogram", lambda: largest_rectangle([2, 0, 2]), 2)
check("wide low rectangle wins: [6, 2, 5, 4, 5, 1, 6]", lambda: largest_rectangle([6, 2, 5, 4, 5, 1, 6]), 12)
check("longer input (2,000 equal bars)", lambda: largest_rectangle([1] * 2_000), 2_000)

<details>
<summary><b>Show solution</b></summary>

```python
def largest_rectangle(heights):
    # Approach: a stack of (start index, height) for bars in increasing height order.
    # When a bar is shorter than the stack top, the top bar can extend no further right,
    # so its rectangle is final: width = current index - its start.
    # A sentinel height of 0 at the end flushes every remaining bar.
    # Complexity: O(n) time, O(n) space.
    stack = []
    best = 0
    for index, height in enumerate(heights + [0]):   # the 0 sentinel empties the stack
        start = index
        while stack and stack[-1][1] >= height:
            start, top_height = stack.pop()
            best = max(best, top_height * (index - start))
        stack.append((start, height))   # this bar can start as far left as the last popped bar
    return best
```

</details>

## Score

In [ ]:
score()